# 03 — What fee each policy actually charged

Net result says how a policy did; this says what it _did_. The question is
whether a policy adapts at all, or sits pinned against one of its own bounds.

**Needs:** the retained swap traces, `results/*.jsonl`, written by the matrix run.

**Units.** Uniswap v4 stores the LP fee as `uint24` in hundredths of a basis
point, so `3000` is 30 bps and `1e6` is 100%. Everything below is converted to
basis points and labelled as such — the paper's Fig. 5 currently labels pip
values as basis points, which is the mistake this notebook must not repeat.

> **The per-swap traces this notebook reads no longer exist.** The UU matrix run
> wrote its own traces into `results/` under byte-identical seven-field
> filenames, overwriting the arbitrage-only experiment's, and `results/` is
> gitignored (2026-08-10; see `PROGRESS.md`). `results/summary.csv`,
> `results/analysis.csv` and every already-drawn figure with its `.csv` table
> view are unaffected — what is gone is the ability to re-derive *per-swap*
> behaviour for the arb-only experiment. The cells that need traces detect this
> and skip with an explanation rather than drawing a figure from the three
> ad-hoc probes that happen to survive the filter.
>
> For the same analysis on the **UU** experiment, see
> `08-uu-flow-behaviour.ipynb`.


In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
sys.path.insert(0, str(ROOT))

from experiments.events import FEE_BOXES_BPS, applied_fees
from experiments.figures import fee_distribution

RESULTS = ROOT / "results"
FIGURES = ROOT / "paper" / "Image"
FIGURES.mkdir(parents=True, exist_ok=True)

def matrix_traces(directory) -> list:
    """Per-swap traces with the exact seven-field matrix name.

    The sensitivity sweeps add an eighth field and must never be pooled in: a
    prefix match once put 62k `captureShare` swaps into a pre-registered figure.
    """
    return [p for p in Path(directory).glob("*.jsonl") if len(p.stem.split("-")) == 7]


TRACES = ROOT / "results"
_found = matrix_traces(TRACES)
# A handful means the ad-hoc probes that survived the filter, not the matrix.
HAVE_TRACES = len(_found) > 1000
print(f"{len(_found)} seven-field traces under {TRACES.name}/ -> usable: {HAVE_TRACES}")
if not HAVE_TRACES:
    print(
        "\nThe arbitrage-only per-swap traces are gone. The UU matrix run wrote\n"
        "its own traces into results/ under byte-identical seven-field names\n"
        "(2026-08-10, PROGRESS.md), and results/ is gitignored, so no copy\n"
        "exists. Unaffected: results/summary.csv, results/analysis.csv, and\n"
        "every already-drawn figure with the CSV table view beside it.\n"
        "Lost: re-deriving PER-SWAP behaviour for the ARB-ONLY experiment.\n"
        "Cells needing traces are skipped below and say so.\n"
        "For the UU experiment's fee behaviour see 08-uu-flow-behaviour.ipynb."
    )

# Reads every trace; about a minute. The parquet beside it is the cached view.
fees = applied_fees(TRACES) if HAVE_TRACES else pd.DataFrame(
    columns=["policy", "pair", "fee_bps"]
)
if HAVE_TRACES:
    fees.to_parquet(TRACES / "applied_fees.parquet")
print(f"{len(fees)} swaps across {fees['pair'].nunique()} pairs")
fees.head(3)

3 seven-field traces under results/ -> usable: False

The arbitrage-only per-swap traces are gone. The UU matrix run wrote
its own traces into results/ under byte-identical seven-field names
(2026-08-10, PROGRESS.md), and results/ is gitignored, so no copy
exists. Unaffected: results/summary.csv, results/analysis.csv, and
every already-drawn figure with the CSV table view beside it.
Lost: re-deriving PER-SWAP behaviour for the ARB-ONLY experiment.
Cells needing traces are skipped below and say so.
For the UU experiment's fee behaviour see 08-uu-flow-behaviour.ipynb.
0 swaps across 0 pairs


,policy,pair,fee_bps


`USDC/USDT` is absent: it produced no swaps at all, consistent with its
zero-trade cells in notebook 02.

Note also what this deliberately excludes. `applied_fees` matches trace
filenames on exactly seven fields, so the `captureShare` sensitivity runs
(notebook 04), which add an eighth, cannot leak in. They are a separate,
non-pre-registered family and must not enter this figure.


## Each policy has its own fee box

This matters more than it looks. `MEVChargeHook` caps at 1000 bps, an order of
magnitude above every other hook, and the static levels have no box at all.
Judged against a single shared 1–100 bps box, `MEVChargeHook` reads as 90% "at
cap" on ETH/SHIB; against its real box it is under 1%.


In [2]:
pd.DataFrame(
    [
        {"policy": p, "floor_bps": lo, "cap_bps": hi}
        for p, (lo, hi) in FEE_BOXES_BPS.items()
    ]
).set_index("policy")

,floor_bps,cap_bps
policy,,
ABHook,1.0,59.0
BAHook,1.0,100.0
DAHook,1.0,100.0
VolatilityHook,1.0,100.0
PegCapture,1.0,100.0
PegDefence,1.0,100.0
MEVChargeHook,30.0,1000.0
MEVChargeHookFixed,30.0,1000.0


In [3]:
if not HAVE_TRACES:
    print("skipped: needs the arbitrage-only per-swap traces -- see the first cell")
else:
    def pinning(group):
        low, high = FEE_BOXES_BPS.get(group.name[0], (None, None))
        values = group["fee_bps"]
        return pd.Series(
            {
                "n": len(values),
                "p5": values.quantile(0.05),
                "median": values.median(),
                "p95": values.quantile(0.95),
                "at_floor": (values <= low).mean() if low else 0.0,
                "at_cap": (values >= high).mean() if high else 0.0,
            }
        )


    table = fees.groupby(["policy", "pair"], group_keys=True).apply(pinning).round(3)
    table.sort_values("at_floor", ascending=False)


skipped: needs the arbitrage-only per-swap traces -- see the first cell


Two things fall out:

- **`PegDefence` is pinned at its 1 bps floor for 100% of swaps on both pairs.**
  It never adapts once in the entire matrix. That is the mechanism behind its
  worst-in-class net result, not bad luck.
- **`MEVChargeHook` splits by pair**: 98% at its 30 bps floor on ETH/USDC, but a
  median of 236 bps on ETH/SHIB.

`ABHook` is the opposite failure mode — it moves, but only across about 9 bps
around 30, which is why it cannot separate itself from static 30 bps.


In [4]:
if not HAVE_TRACES:
    print("skipped: needs the arbitrage-only per-swap traces -- see the first cell")
else:
    from IPython.display import Image, display

    fee_distribution(fees, FIGURES / "fee_distribution.pdf")
    fee_distribution(fees, FIGURES / "fee_distribution.png")
    display(Image(filename=str(FIGURES / "fee_distribution.png")))


skipped: needs the arbitrage-only per-swap traces -- see the first cell


Quantile intervals rather than histograms, because four of these policies are
exact constants: a histogram renders a constant as a one-pixel spike that reads
as an empty panel. A constant is honestly a dot.


## What each policy did through one window

The distribution above says how often a fee was charged; it does not say
_when_. This traces the most volatile ETH/SHIB window in the year — chosen by
the volatility statistic, not by eye — and shows the price path with every
policy's response underneath it on the same clock.

The fee axis is shared and logarithmic on purpose. These policies span three
decades, and giving each panel its own scale would draw a 0.3 bps wobble the
same size as a 300 bps swing. On a shared axis the policies that barely move
look like they barely move.


In [5]:
if not HAVE_TRACES:
    print("skipped: needs the arbitrage-only per-swap traces -- see the first cell")
else:
    from experiments.binance import fetch_klines
    from experiments.events import fee_trajectories
    from experiments.export_trace import align_traces
    from experiments.figures import fee_response
    from experiments.matrix import load_segments

    _, selected = load_segments()
    window = selected["ETH/SHIB"].sort_values("volatility").iloc[-1]
    start, end = int(window.start_ms), int(window.end_ms)
    print(
        f"window {pd.to_datetime(start, unit='ms').date()}, volatility {window.volatility:.4f}"
    )

    # The reference price the hooks read, rebuilt from the same cache the run used.
    eth = fetch_klines("ETHUSDT", start, end, ROOT / "data")
    shib = fetch_klines("SHIBUSDT", start, end, ROOT / "data")
    eth, shib = align_traces(eth, shib)
    ratio = eth["close"].to_numpy() / shib["close"].to_numpy()
    prices = pd.Series(ratio / ratio[0], index=range(len(ratio)))

    trajectories = fee_trajectories(ROOT / "results", ("ETHUSDT", "SHIBUSDT"), start)
    print(f"{len(trajectories)} swaps across {trajectories['policy'].nunique()} policies")


skipped: needs the arbitrage-only per-swap traces -- see the first cell


In [6]:
if not HAVE_TRACES:
    print("skipped: needs the arbitrage-only per-swap traces -- see the first cell")
else:
    fee_response(trajectories, prices, FIGURES / "fee_response.pdf")
    fee_response(trajectories, prices, FIGURES / "fee_response.png")
    display(Image(filename=str(FIGURES / "fee_response.png")))


skipped: needs the arbitrage-only per-swap traces -- see the first cell


### One policy at its own scale

The shared log axis above keeps six policies comparable and therefore flattens
anything moving by less than a decade. For a single policy a linear axis fitted
to its own range shows the adjustment step by step.

x is trade index, not clock time: a hook revises its fee only when it is
called, so successive trades are its actual step sequence.

**On units.** The y-axis is basis points, converted from the `uint24` pip value
the pool stores. The paper's current Fig. 5 plots raw pip values (3000–3200)
under a "bps" label, which reads as a 30% fee where the pool charges 30 bps.


In [7]:
if not HAVE_TRACES:
    print("skipped: needs the arbitrage-only per-swap traces -- see the first cell")
else:
    from experiments.figures import fee_response_detail

    #  Policy("BAHook", "BAHook"),
    #     Policy("DAHook", "DAHook"),
    #     Policy("ABHook", "ABHook"),
    #     Policy("PegDefence", "PegDefence"),
    #     Policy("PegCapture", "PegCapture"),
    #     Policy("MEVChargeHook", "MEVChargeHook", size_dependent=True),
    #     # The same template with a dimensionless impact measure. Both are run: the
    #     # shipped one sizes a trade by amount/L, which is not a ratio, and charges
    #     # 354 bps one way against 30 bps the other purely from the input's
    #     # denomination. See MEVChargeHookFixed.
    #     Policy("MEVChargeHookFixed", "MEVChargeHookFixed", size_dependent=True),
    #     Policy("VolatilityHook",

    for name in (
        "DAHook",
        "ABHook",
        "PegDefence",
        "PegCapture",
        "MEVChargeHook",
        "MEVChargeHookFixed",
        "VolatilityHook",
        "MyHook@3000",
    ):
        fee_response_detail(trajectories, prices, name, FIGURES / f"fee_detail_{name}.pdf")
        fee_response_detail(trajectories, prices, name, FIGURES / f"fee_detail_{name}.png")
        display(Image(filename=str(FIGURES / f"fee_detail_{name}.png")))


skipped: needs the arbitrage-only per-swap traces -- see the first cell
